# Testing the finetuned model (qwen)

This notebook tests the finetuned model (trained with 2 000 input-output pairs) saved at the end of the finetuning notebook. The steps are:

1. **Define a test conversation** — a doctor-patient dialogue not used in finetuning
2. **Load libraries and set sampling parameters** — import vLLM and configure generation settings
3. **Build function** - function to format the conversation into the chat structure the model expects
4. **Configurations** - set paths and variables
5. **Load the model** — point to the merged finetuned model and load it with vLLM
6. **Test with system prompt** — run inference with the clinical note instructions included, as the model was trained
7. **Test without system prompt** — run the same conversation without instructions to see how the model behaves without guidance
8. **Compare outputs** — observe the difference between the two runs to understand the effect of finetuning and prompting

## 1. Define test conversation

This is the doctor-patient conversation we will use to test the finetuned model. It describes a patient presenting with lower right abdominal pain — a case that may indicate appendicitis.

In [ ]:
prompt = """Doctor: Hey, how are you doing today?

Patient: Hello doctor. I am feeling pain on the bottom right in my belly.

Doctor: How long has the pain been there?

Patient: It started yesterday evening and got worse during the night.

Doctor: Can you describe the pain? Is it sharp, dull, cramping, or something else?

Patient: It started as a dull ache, but now it feels sharp when I move or walk.

Doctor: On a scale from 1 to 10, how strong is the pain?

Patient: Around 7 out of 10.

Doctor: Have you noticed any nausea, vomiting, fever, or changes in appetite?

Patient: Yes, I feel nauseous and I did not want breakfast this morning. I also think I have a slight fever.

Doctor: Have you had diarrhea or constipation?

Patient: No diarrhea, but I have not gone to the bathroom since yesterday.

Doctor: Does anything make the pain better or worse?

Patient: Moving makes it worse. Lying still helps a little.

Doctor: Have you experienced this kind of pain before?

Patient: No, never this bad.

Doctor: Do you have any medical conditions or take any medications regularly?

Patient: No major medical conditions. I only take allergy medicine sometimes.

Doctor: Thank you. I would like to examine your abdomen now, especially the lower right side.

Patient: Okay.

Doctor: When I press here, does it hurt?

Patient: Yes, especially when you let go.

Doctor: I understand. Based on your symptoms and the examination, this could be appendicitis. I recommend blood tests and an abdominal scan as soon as possible.

Patient: Is it serious?

Doctor: It can become serious if untreated, but we caught it early. We will arrange further testing immediately.

Patient: Thank you, doctor.

Doctor: You're welcome. We will take good care of you."""

## 2. Import libraries and set sampling parameters

We use **vLLM** for inference — a high-performance library optimised for running large language models efficiently on GPU. It handles batching, memory management, and generation internally.

`temperature=0.0` means fully deterministic output — the model always picks the most likely next token, producing consistent and reproducible results. 

In [ ]:
from vllm import LLM, SamplingParams
import os

import warnings
warnings.filterwarnings("ignore")

In [ ]:
SAMPLING = dict(temperature=0.0, max_tokens=1000)
sampling_params = SamplingParams(**SAMPLING)

## 3. Build function

This function formats the conversation into the chat structure the model expects. It supports two modes controlled by `use_system_prompt`:
- **`True`** — includes the system prompt that instructs the model to produce a structured clinical note, mimicking how the model was finetuned
- **`False`** — sends only the raw conversation with no instructions, to see how the base model responds without guidance

In [ ]:
def build_messages(example: str, use_system_prompt: bool = True) -> list[dict]:
    messages = [
        {"role": "system",
        "content": """You are a medical clinical documentation assistant. 
You task is to convert a dialogue between a doctor and patient into a structured clinical note in the following output format:
REASON FOR VISIT:
<Brief summary of why the patient is seeking care>
PATIENT DETAILS AND HISTORY:
<Age, gender, relevant demographics, relevant past medical history, conditions, medications, surgeries, lifestyle factors>
CURRENT STATUS:
<Current symptoms, findings, vitals, clinical observations>
TREATMENTS/ACTIONS:
<Medications prescribed, procedures performed, advice given>
FOLLOW-UP PLAN:
<Next steps, monitoring, referrals, timelines. Follow-up plan should not include "future" details that are mentioned in the note, but rather should infer what the next steps would be based on the found future details.>
"""},
        {"role": "user",   "content": example},
    ]
    if not use_system_prompt:
        messages = [messages[-1]]

    return messages

## 4. Configurations

Set the path to the merged finetuned model saved at the end of the finetuning notebook. Make sure the `input_model` and `model_output_name` match exactly what was used there.

In [ ]:
SLURM_JOB_ACCOUNT = os.getenv("SLURM_JOB_ACCOUNT")
USER = os.getenv("SLURM_JOB_USER")
output_path = f"/scratch/{SLURM_JOB_ACCOUNT}/{USER}/health_case/ft_model"
input_model = "Qwen/Qwen3-4B-Instruct-2507"
model_output_name = f"{input_model}_finetuned"
merged_output_dir = os.path.join(
    output_path,
    f"{model_output_name}_merged"
)

print(f"Merged model path: {merged_output_dir}")

## 5. Load the finetuned model

Load the merged finetuned model using vLLM. `tensor_parallel_size=1` means the model runs on a single GPU. This may take a minute.

You will see a number of INFO messages printed during loading — these are normal vLLM startup logs describing internal configuration such as scheduling mode, GPU kernel selection, and memory allocation. You can ignore these. The model is ready when the cell finishes executing.

In [ ]:
llm = LLM(model=merged_output_dir, tensor_parallel_size=1, dtype="bfloat16")

## 6. Inference with system prompt

Run inference with the system prompt included. This is how the model was trained — it receives clear instructions about the expected output format. The output should be a well-structured clinical note.

In [ ]:
outputs = llm.chat(build_messages(prompt), sampling_params, use_tqdm=True)

In [ ]:
from IPython.display import Markdown, display

display(Markdown("---"))
display(Markdown("### Model response woth system prompt:"))
display(Markdown(outputs[0].outputs[0].text))
display(Markdown("---"))

## 7. Inference without system prompt

Run the same conversation without the system prompt to see how the model behaves without explicit instructions. Compare this output to Test 1 — the difference illustrates the effect of finetuning and prompting on the model's output format and clinical accuracy.

In [ ]:
outputs = llm.chat(build_messages(prompt, use_system_prompt=False), sampling_params, use_tqdm=True)

In [ ]:
from IPython.display import Markdown, display

display(Markdown("---"))
display(Markdown("### Model response without system prompt:"))
display(Markdown(outputs[0].outputs[0].text))
display(Markdown("---"))